# Kaggriculture — replay → edge-labeled imitation dataset

Builds the September 2026 dataset on Kaggle: downloads episode replays,
converts each to a format-1 shard (`FORMAT.md`), adds transition-matched edge
labels, and writes everything to `/kaggle/working/kaggle_shards`.

CHECKPOINTED: finished episodes are skipped on re-run, the report is rewritten
every 25 episodes — a stopped notebook resumes from where it stopped.

Code source: the attached dataset `amirelite/kaggriculture-edge-dataset-code`
(the repo's `agent/` + the two builders, zipped — the repo itself is private).

In [ ]:
# 1. attach the code dataset and unpack the CURRENT bundle
# (the dataset may carry the tree as files OR as code_bundle.zip; handle both)
import os, shutil, glob, zipfile, subprocess
WORK = "/kaggle/working"
CANDIDATES = glob.glob("/kaggle/input/*edge-dataset-code*")
assert CANDIDATES, "code dataset not attached: add it in Input -> Add Input"
CODE = CANDIDATES[0]
zfs = glob.glob(f"{CODE}/**/code_bundle.zip", recursive=True)
if zfs:
    with zipfile.ZipFile(zfs[0]) as z:
        z.extractall(WORK)
else:
    shutil.copytree(CODE, WORK, dirs_exist_ok=True)
os.chdir(WORK)
subprocess.run(["pip", "install", "-q", "kaggle-environments==1.32.7"], check=True)  # the graph artifact was built against 1.32.7; the notebook image ships 1.29.3
print("code ready:", os.path.isdir("agent/artifact"), os.path.isdir("offline_lab/build"))

In [ ]:
# 2. the month's episode list: episodes of the user's OWN submissions
# (ListEpisodes requires a submissionId; the leaderboard's teams are not ours)
import requests, os
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

def list_episodes(month="2026-09"):
    """Episodes of every own submission that played in `month`.

    ListEpisodes needs a submissionId (a teamId is rejected); the submission
    list provides them. Each episode entry carries BOTH agents' submissionIds
    and rewards — the opponent seat's id is what widens the pool.
    """
    subs = api.competition_submissions("kaggriculture")
    sub_ids = []
    for s in subs:
        d = s if isinstance(s, dict) else getattr(s, "__dict__", {})
        ref = d.get("_ref") or d.get("ref")
        if ref:
            sub_ids.append(int(ref))
    out = {}
    url = "https://www.kaggle.com/api/i/competitions.EpisodeService/ListEpisodes"
    for sid in sub_ids:
        token = None
        for _ in range(8):
            body = {"submissionId": int(sid), "pageSize": 100}
            if token:
                body["pageToken"] = token
            r = requests.post(url, json=body)
            if r.status_code != 200:
                break
            data = r.json()
            for ep in data.get("episodes", []):
                if str(ep.get("createTime", "")).startswith(month):
                    out[ep["id"]] = ep.get("createTime")
            token = data.get("nextPageToken")
            if not token:
                break
    return out

EPISODES = list_episodes()
print(f"September 2026 episodes: {len(EPISODES)}")
EPISODE_IDS = sorted(EPISODES)

In [ ]:
# 3. download + convert + label, CHECKPOINTED per episode
import sys, time, json, os, requests
from pathlib import Path
import numpy as np
sys.path.insert(0, "/kaggle/working")

from offline_lab.build.kaggle_dataset import convert_replay, Vocab
from offline_lab.build.add_edge_labels import _label_shard_arrays
from agent.planner.inputs import GRAPH_PATH as _GRAPH_PATH
from agent.tile_dp.graph import TileGraph

WORKDIR = Path("/kaggle/working/kaggle_shards")
RAW = Path("/kaggle/tmp/replays")
WORKDIR.mkdir(parents=True, exist_ok=True); RAW.mkdir(parents=True, exist_ok=True)

graph = TileGraph.load(_GRAPH_PATH)
voc = Vocab()

done = {p.stem for p in WORKDIR.glob("*.npz")}
report_path = WORKDIR / "build_report.json"
report = json.loads(report_path.read_text()) if report_path.exists() else {"shards": {}}
print(f"resume: {len(done)} shards on disk, {len(EPISODE_IDS)-len(done)} to go")

for k, eid in enumerate(EPISODE_IDS):
    if str(eid) in done:
        continue
    t0 = time.time()
    try:
        target = RAW / f"episode-{eid}-replay.json"
        if not target.exists():
            # the notebook's bundled kaggle SDK chokes on chunked replay
            # responses (KeyError: content-length); this POST needs nothing
            # but requests + the credentials the notebook environment carries
            from kagglesdk.competitions.types.competition_api_service import (
                ApiGetEpisodeReplayRequest)
            with api.build_kaggle_client() as kag:
                req = ApiGetEpisodeReplayRequest()
                req.episode_id = int(eid)
                resp = kag.competitions.competition_api_client.get_episode_replay(req)
                if resp.status_code != 200:
                    raise RuntimeError(f"replay HTTP {resp.status_code} for {eid}")
                with open(target, "wb") as fh:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        fh.write(chunk)
        shard = convert_replay(target, graph.key_index, voc)
        if shard is not None:
            # WINNERS-ONLY: keep samples of seats whose agent won this episode
            # (rewards are the final scores; ties keep both seats). Losing seats
            # would poison the imitation labels with losing play.
            _rewards = shard["reward"]
            _winner_mask = _rewards >= _rewards.max()
            shard = {k: (v[_winner_mask] if hasattr(v, "shape") and v.shape[:1] == _winner_mask.shape else v)
                     for k, v in shard.items()}
            # ragged rows must be re-sliced for the surviving seats: labels/orders/ops
            # live per sample; rebuild ptr arrays from per-sample counts BEFORE filtering
            # (convert_replay already stored flat arrays; we recover counts from ptr)
            def _reslice(ptr_key, cols, flat_key_prefix):
                ptr = shard[ptr_key]
                keep_rows = []
                for i in range(len(_winner_mask)):
                    if _winner_mask[i]:
                        keep_rows += list(range(int(ptr[i]), int(ptr[i+1])))
                shard[flat_key_prefix] = shard[flat_key_prefix][keep_rows]
                new_ptr = np.zeros(int(_winner_mask.sum()) + 1, dtype=np.int64)
                for j, i in enumerate([i for i in range(len(_winner_mask)) if _winner_mask[i]]):
                    new_ptr[j+1] = new_ptr[j] + (int(ptr[i+1]) - int(ptr[i]))
                shard[ptr_key] = new_ptr
            for pk, fk in [("op_ptr", "op"), ("ord_ptr", "ord"), ("lab_ptr", "lab")]:
                _reslice(pk, None, fk)
            voc_list = [voc.strings[i] for i in range(len(voc.strings))]
            entity_codes = {n: voc.code(n) for n in voc.strings}
            steps = json.load(open(target))["steps"]
            arrays, summary = _label_shard_arrays(shard, steps, graph, voc_list, entity_codes)
            shard.update(arrays)
            for kk, vv in summary.items():
                shard[f"lab_{kk}"] = np.int64(vv)
            np.savez_compressed(WORKDIR / f"{eid}.npz", **shard)
            report["shards"][str(eid)] = {
                "samples": int(shard["n_samples"]),
                "unknown": int(shard["unknown_keys"]),
                "matched": int(summary["matched"]),
                "out_of_graph": int(summary["out_of_graph"]) + int(summary["opset_gap"]),
                "secs": round(time.time()-t0, 1)}
        target.unlink(missing_ok=True)
    except Exception as e:
        report["shards"][str(eid)] = {"error": repr(e)[:200]}
        print(f"{eid}: ERROR {repr(e)[:120]}")
    if k % 25 == 0:
        report_path.write_text(json.dumps(report))
        print(f"{k}/{len(EPISODE_IDS)} processed, {len(list(WORKDIR.glob('*.npz')))} shards")

voc.save(WORKDIR / "vocab.json")
report_path.write_text(json.dumps(report, indent=1))
print("BUILD COMPLETE:", len(list(WORKDIR.glob('*.npz'))), "shards")

In [ ]:
# 4. summary
import numpy as np
tot = {"samples": 0, "matched": 0, "no_action": 0, "out_of_graph": 0, "phantom": 0}
for p in WORKDIR.glob("*.npz"):
    sh = np.load(p)
    tot["samples"] += int(sh["n_samples"])
    for key in ("matched", "no_action", "out_of_graph"):
        tot[key] += int(sh.get(f"lab_{key}", 0))
    tot["phantom"] += int(sh.get("lab_phantom_total", 0))
print(tot)
den = tot["matched"] + tot["no_action"] + tot["out_of_graph"]
print(f"labeled-edge coverage: {100*tot['matched']/max(1,den):.2f}%")